In [1]:
from pathlib import Path
import json

from sklearn import metrics
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
from scipy.spatial.distance import cosine
from scipy.stats import skew, skewtest, kurtosis, kurtosistest
from sklearn.metrics.pairwise import cosine_similarity

from lib.pose_utils import get_poem_embedding
from mime_db import MimeDb

# Connect to the database
db = await MimeDb.create()

In [ ]:
# This cell is just to explore the data available for one video -- usually skipped

VIDEO_FILE = "A_Letter_to_My_Nephew.mp4"  # Just the name of the video file, no path

video_path = Path("videos", VIDEO_FILE)

# Get video metadata
video_name = video_path.name
print(video_name)
video_id = await db.get_video_id(video_name)

print("VIDEO ID", video_id)

video_data = await db.get_video_by_id(video_id)
video_fps = video_data["fps"]
video_frame_count = video_data["frame_count"]

video_seconds = video_frame_count / video_fps

video_poses = await db.get_pose_data_from_video(video_id)
video_hands = await db.get_hand_data_from_video(video_id)

In [2]:
def get_distribution_stats(distrib, plot=False):

    if len(distrib) == 0:
        return {
            "count": 0,
            "mean": 0,
            "median": 0,
            "stdev": 0,
            "skewness": 0,
            "kurtosis": 0,
        }

    if skewtest(distrib).pvalue < 0.05:
        skewness = skew(distrib)
    else:
        skewness = 0

    if kurtosistest(distrib).pvalue < 0.05:
        kurtosis_value = kurtosis(distrib)
    else:
        kurtosis_value = 0

    if plot:
        plt.hist(distrib, bins="auto")  # arguments are passed to np.histogram
        plt.show()

    return {
        "count": len(distrib),
        "mean": np.mean(distrib),
        "median": np.median(distrib),
        "stdev": np.std(distrib),
        "skewness": skewness,
        "kurtosis": kurtosis_value,
    }


stylometry_videos = [
    "A_Letter_to_My_Nephew.mp4",
    "Analogy-Dora_Tramontane.mp4",
    "A_Quarelling_Pair-2007_upscaled.mp4",
    "BillTJones_Analogy_Ambros_The_Emigrant.mp4",
    "D-ManintheWaters.mp4",
    "FondlyDoWeHope-Fervently_Do_We_Pray.mp4",
    "Holzer_DuetTruisms_upscaled.mp4",
    "PlayandPlay.mp4",
    "Secret_Pastures.mp4",
    "Story-Time.mp4",
    "WeShallNotBeMoved_upscaled.mp4",
    "A_Balsa_da_Medusa-oratorio_de_Hans_Werner_Henze_Opera_Holandesa.mp4",
    "Democracy_in_America_upscaled.mp4",
    "Don_Giovanni_Mozart.mp4",
    "Go_Down_Moses_2014.mp4",
    "Inferno_Dante.mp4",
    "Parsifal-Wagner-La_Monnaie-De_Munt.mp4",
    "Purgatorio-Dante_upscaled.mp4",
    "Requiem-Mozart-Festival_dAix-en-Provence.mp4",
    "Resurrection-Mahler.mp4",
    "The_Magic_Flute_Mozart_La_Monnaie_De_Munt.mp4",
    "Bluebeard_Voix_Humaine.mp4",
    "Die_Gezeichneten.mp4",
    "Elektra_Strauss.mp4",
    "Ifigenia_emTauris-Gluck.mp4",
    "Lady_Macbeth_of_Mtsensk_Districtp1.mp4",
    "Lady_Macbeth_of_Mtsensk_Districtp2.mp4",
    "Medeia-Luigi_Cherubini.mp4",
    "Tales_of_Hoffman.mp4",
    "the_french.mp4",
    "Wozzeck_Alban_Berg.mp4",
]

director_videos = {}
video_directors = {}
director_videos["BillTJones"] = [
    "A_Letter_to_My_Nephew.mp4",
    "Analogy-Dora_Tramontane.mp4",
    "A_Quarelling_Pair-2007_upscaled.mp4",
    "BillTJones_Analogy_Ambros_The_Emigrant.mp4",
    "D-ManintheWaters.mp4",
    "FondlyDoWeHope-Fervently_Do_We_Pray.mp4",
    "Holzer_DuetTruisms_upscaled.mp4",
    "PlayandPlay.mp4",
    "Secret_Pastures.mp4",
    "Story-Time.mp4",
    "WeShallNotBeMoved_upscaled.mp4",
]
director_videos["Castellucci"] = [
    "A_Balsa_da_Medusa-oratorio_de_Hans_Werner_Henze_Opera_Holandesa.mp4",
    "Democracy_in_America_upscaled.mp4",
    "Don_Giovanni_Mozart.mp4",
    "Go_Down_Moses_2014.mp4",
    "Inferno_Dante.mp4",
    "Parsifal-Wagner-La_Monnaie-De_Munt.mp4",
    "Purgatorio-Dante_upscaled.mp4",
    "Requiem-Mozart-Festival_dAix-en-Provence.mp4",
    "Resurrection-Mahler.mp4",
    "The_Magic_Flute_Mozart_La_Monnaie_De_Munt.mp4",
]
director_videos["Warlikowski"] = [
    "Bluebeard_Voix_Humaine.mp4",
    "Die_Gezeichneten.mp4",
    "Elektra_Strauss.mp4",
    "Ifigenia_emTauris-Gluck.mp4",
    "Lady_Macbeth_of_Mtsensk_Districtp1.mp4",
    "Lady_Macbeth_of_Mtsensk_Districtp2.mp4",
    "Medeia-Luigi_Cherubini.mp4",
    "Tales_of_Hoffman.mp4",
    "the_french.mp4",
    "Wozzeck_Alban_Berg.mp4",
]
director_videos["Other"] = []
director_ids = {"BillTJones": 0, "Castellucci": 1, "Warlikowski": 2}
stylometry_directors = ["BillTJones", "Castellucci", "Warlikowski"]

for director in director_videos:
    for video_name in director_videos[director]:
        video_directors[video_name] = director_ids[director]

video_labels = [video_directors[video_name] for video_name in stylometry_videos]

with open("archetypes/pose_archetypes.json", "r") as poses_file:
    pose_archetypes = json.load(poses_file)
with open("archetypes/hand_archetypes.json", "r") as hands_file:
    hand_archetypes = json.load(hands_file)

pose_descriptions = [arch["description"] for arch in pose_archetypes] # [:10] # Only use 3x3 + 1 poses

hand_descriptions = [arch["description"] for arch in hand_archetypes]


In [ ]:
# Used to add the pose embeddings to the pose archetypes (they're originally just
# available as global 3D coords) and export them to a JSON file to be used at
# various places in the site.

import copy

full_pose_archetypes = copy.deepcopy(pose_archetypes)

for a, archetype in enumerate(pose_archetypes):
    flattened_global3d = [c for i, c in enumerate(archetype["global3d_coco13"]) if (i + 1) % 3 != 0]
    poem_embedding = get_poem_embedding(flattened_global3d)
    full_pose_archetypes[a]["poem_embedding"] = poem_embedding

with open("full_pose_archetypes.json", "w", encoding="utf-8") as outfile:
    json.dump(full_pose_archetypes, outfile, indent=4)

pose_archetypes = full_pose_archetypes

In [ ]:
# Get the prevalence of a pose archetype in a video
# Given a pose archetype
# For every pose in the video
#   Calculate its cosine similarity to the archetype, put it into an array
# Average the similarities (maybe derive some other statistics too)
def get_fingerprints(archetypes, video_poses_or_hands, comparison_metric):
    mean_sims = [] # Could do medians as well/instead, but it doesn't make much difference
    for archetype in archetypes: # [:10]
        sims = []
        if comparison_metric == "poem_embedding":
            flattened_global3d = [c for i, c in enumerate(archetype["global3d_coco13"]) if (i + 1) % 3 != 0]
            archetype_vector = get_poem_embedding(flattened_global3d)
        else:
            archetype_vector = archetype[comparison_metric]
        for pose_or_hand in video_poses_or_hands:
            sims.append(1 - cosine(archetype_vector, pose_or_hand[comparison_metric]))

        mean_sims.append(np.mean(sims))

    return mean_sims

if not os.path.isfile("archetypes/archetype_fingerprints.json"):
    delsarte_pose_fingerprints = {}
    for video_name in stylometry_videos:
        print("Computing pose fingerprints for", video_name)
        video_id = await db.get_video_id(video_name)
        video_poses = await db.get_pose_data_from_video(video_id)
        delsarte_pose_fingerprints[str(video_id)] = {"video_name": video_name, "poem_embedding": get_fingerprints(pose_archetypes, video_poses, "poem_embedding"), "global3d_coco13": get_fingerprints(pose_archetypes, video_poses, "global3d_coco13")}

    delsarte_hand_fingerprints = {}
    for video_name in stylometry_videos:
        print("Computing hand fingerprints for", video_name)
        video_id = await db.get_video_id(video_name)
        video_hands = await db.get_hand_data_from_video(video_id)
        delsarte_hand_fingerprints[str(video_id)] = {"video_name": video_name, "joint_angles3d": get_fingerprints(hand_archetypes, video_hands, "joint_angles3d"), "class_weights": get_fingerprints(hand_archetypes, video_hands, "class_weights")}
else:
    with open("archetypes/archetype_fingerprints.json", "r", encoding="utf-8") as archfile:
        archetype_data = json.load(archfile)
        delsarte_pose_fingerpints = archetype_data["pose"]
        delsarte_hand_fingerprints = archetype_data["hand"]

In [ ]:
all_fingerprints = {"poses": delsarte_pose_fingerprints, "hands": delsarte_hand_fingerprints}

with open("archetypes/archetype_fingerprints.json", "w", encoding="utf-8") as outfile:
    json.dump(all_fingerprints, outfile, indent=4)

# This determines whether the cells below consider poses or hands
# delsarte_fingerprints = delsarte_pose_fingerprints
# delsarte_fingerprints = delsarte_hand_fingerprints

In [ ]:
# "Dumbest" possible classifier, using "leave one out" approach and cosine similarity

hits = 0
misses = 0

predictions = []

metric = "global3d_coco13" # either poem_embedding or global3d_coco13 for poses, or joint_angles3d or class_weights for hands

for i, video_name in enumerate(stylometry_videos):

    fingerprints_by_director = {
        "BillTJones": [],
        "Castellucci": [],
        "Warlikowski": [],
    }

    for e, comparison_video in enumerate(stylometry_videos):
        if e == i:
            continue

        video_director = stylometry_directors[video_directors[comparison_video]]
        
        fingerprints_by_director[video_director].append(delsarte_fingerprints[comparison_video][metric])

    billtjones_avg_fingerprint = np.mean(
        fingerprints_by_director["BillTJones"], axis=0
    )
    castellucci_avg_fingerprint = np.mean(
        fingerprints_by_director["Castellucci"], axis=0
    )
    warlikowski_avg_fingerprint = np.mean(
        fingerprints_by_director["Warlikowski"], axis=0
    )

    directors_matrix = np.stack(
        (
            billtjones_avg_fingerprint,
            castellucci_avg_fingerprint,
            warlikowski_avg_fingerprint,
        ),
        axis=0,
    )

    video_fingerprint = delsarte_fingerprints[video_name][metric]
    true_label = video_directors[video_name]

    sims = cosine_similarity(directors_matrix, np.array([video_fingerprint]))

    pred_label = np.argmax(sims)
    print(video_name, "Prediction:", pred_label, "True:", true_label)

    predictions.append(pred_label)

    if pred_label == true_label:
        hits += 1
    else:
        misses += 1

print("Hits:", hits, "Misses:", misses, f"{hits*100/(hits+misses):.2f}%")

confusion_matrix = metrics.confusion_matrix(video_labels, predictions)
cm_display = metrics.ConfusionMatrixDisplay.from_predictions(
    video_labels,
    predictions,
    display_labels=["BillTJones", "Castellucci", "Warlikowski"],
    colorbar=False,
)
plt.title("LOO - pose embedding sims to Delsarte archetypes")
cm_display.plot()
#plt.show()

In [ ]:
# Train a classifier on all but one; see where that one is classified,
# repeat for all videos (leave-one-out).
# Potentially repeat the process for different random seeds, if the
# classification algorithm is nondeterministic

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

import xgboost as xgb
# from sklearn.svm import SVC, LinearSVC, NuSVC
# from sklearn.neural_network import MLPClassifier

N_SEEDS = 10

hits = 0
misses = 0
predicted = []
actual = []

metric = "poem_embedding" # either poem_embedding or global3d_coco13 for poses, or joint_angles3d or class_weights for hands

X = []
for video_name in stylometry_videos:
    X.append(delsarte_fingerprints[video_name][metric])

for r in range(N_SEEDS):
    for h, held_out_vector in enumerate(X):

        X_train = np.delete(X, h, 0)
        y_train = np.delete(video_labels, h, 0)
        # X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.33, random_state=r)

        # clf = MLPClassifier()
        # lf = NuSVC(random_state=42)
        # clf = GaussianNB()
        # clf = xgb.XGBClassifier(tree_method="hist", early_stopping_rounds=2)
        clf = RandomForestClassifier(random_state=r)
        clf.fit(X_train, y_train)
        # clf.fit(X_train,  y_train, eval_set=[(X_test, y_test)])

        # pred = clf.predict(held_out_vector.reshape(1, -1))[0]
        pred = clf.predict([held_out_vector])[0]

        if pred == video_labels[h]:
            hits += 1
        else:
            misses += 1

        actual.append(video_labels[h])
        predicted.append(pred)

        # print("For ", video_names[h], "pred is", pred, "true is", video_labels[h])

print(f"Hits: {hits}, Misses: {misses}, {hits*100/(hits+misses):.2f}%")

confusion_matrix = metrics.confusion_matrix(actual, predicted)
cm_display = metrics.ConfusionMatrixDisplay.from_predictions(
    actual,
    predicted,
    display_labels=["BillTJones", "Castellucci", "Warlikowski"],
    colorbar=False,
)
plt.title("RF - POEM embedding sims to Delsarte archetypes")
cm_display.plot()

In [ ]:
from sklearn.model_selection import cross_validate, cross_val_score, KFold
import random
from datetime import datetime

# Another way to do it
# scoring = ['precision_macro', 'recall_macro']
# clf = RandomForestClassifier(random_state=0)
# scores = cross_validate(clf, X, video_labels, scoring=scoring)
# print("precision scores:", scores["test_precision_macro"], "recall scores:", scores["test_recall_macro"])

metric = "poem_embedding" # either poem_embedding or global3d_coco13 for poses, or joint_angles3d or class_weights for hands

X = []
for video_name in stylometry_videos:
    X.append(delsarte_fingerprints[video_name][metric])

N_SEEDS = 10

print("10-fold cross validation with 10 random seeds for video feature vectors")
all_scores = []
for i in range(N_SEEDS):
    r = random.seed(datetime.now().timestamp())
    clf = RandomForestClassifier(random_state=r)
    # clf = GaussianNB()
    cv = KFold(n_splits=10, random_state=r)
    scores = cross_val_score(
        clf, X, video_labels, scoring="accuracy", cv=cv, n_jobs=-1
    )
    all_scores.extend(scores)
print(
    "Accuracy: %.3f ,\nStandard Deviations :%.3f"
    % (np.mean(all_scores), np.std(all_scores))
)

In [ ]:
from sklearn.inspection import permutation_importance
import time

feature_names = pose_descriptions # or pose_descriptions

metric = "poem_embedding" # either poem_embedding or global3d_coco13 for poses, or joint_angles3d or class_weights for hands

X = []
for video_name in stylometry_videos:
    X.append(delsarte_fingerprints[video_name][metric])

import xgboost as xgb

X_train, X_test, y_train, y_test = train_test_split(
    X, video_labels, test_size=0.33, random_state=42
)

clf = RandomForestClassifier(random_state=0)
# clf = GaussianNB()
# clf = xgb.XGBClassifier(tree_method="hist", early_stopping_rounds=2)

clf.fit(X_train, y_train)
# clf.fit(X_train, y_train, eval_set=[(X_test, y_test)])

start_time = time.time()
perm_importance = permutation_importance(
    clf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=2
)
elapsed_time = time.time() - start_time
print(f"Elapsed time to compute the importances: {elapsed_time:.3f} seconds")

clf_importances = pd.Series(perm_importance.importances_mean, index=feature_names)

# importances = clf.feature_importances_
# importances_std = np.std([tree.feature_importances_ for tree in clf.estimators_], axis=0)
# clf_importances = pd.Series(importances, index=vector_names)

fig, ax = plt.subplots()
clf_importances.plot.bar(yerr=perm_importance.importances_std, ax=ax)
ax.set_title("Archetype importances (permutation) - pose embedding")
ax.set_ylabel("Mean accuracy decrease")
ax.tick_params(axis="x", labelrotation=40)
plt.setp(ax.xaxis.get_majorticklabels(), ha="right")
fig.tight_layout()
plt.show()

sorted_idx = perm_importance.importances_mean.argsort()

# plt.barh(
#     np.array(feature_names)[sorted_idx],
#     perm_importance.importances_mean[sorted_idx],
#     xerr=perm_importance.importances_std[sorted_idx],
# )
# plt.barh(np.array(vector_names)[sorted_idx], perm_importance.importances_mean[sorted_idx], xerr=perm_importance.importances_std[sorted_idx])
plt.barh(
    np.array(feature_names)[sorted_idx],
    perm_importance.importances_mean[sorted_idx],
    xerr=perm_importance.importances_std[sorted_idx],
)
plt.xlabel("Permutation Importance")
plt.show()

In [ ]:
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

fp_type = "poses"
delsarte_fingerprints = delsarte_pose_fingerprints
descriptions = pose_descriptions
metric = "poem_embedding" # pose: global3d_coco13 or poem_embedding; hand: joint_angles3d or class_weights
archetypes = pose_archetypes
performance = "Don_Giovanni_Mozart.mp4"

def offset_image(x, y, arch, bar_is_too_short, ax):
    img = plt.imread(f"archetypes/{fp_type}/{arch['image_filename']}")
    im = OffsetImage(img, zoom=.09, cmap=mpl.colormaps['gray'])
    im.image.axes = ax
    x_offset = .2
    if bar_is_too_short:
        x = 0
    ab = AnnotationBbox(im, (x, y), xybox=(x_offset, 0), frameon=False,
                        xycoords='data', boxcoords="offset points", pad=0)
    ax.set_facecolor('#FFFFFF')
    ax.add_artist(ab)

fig = plt.figure(figsize=(8,12))

height = .5

fingerprint_data = list(reversed(delsarte_fingerprints[performance][metric]))
plt.barh(list(reversed(descriptions)), fingerprint_data, height=height, align='center', alpha=0.8, color="#663399")
ax = plt.gca()
ax.tick_params(axis="y", labelrotation=40)

max_value = max(fingerprint_data)

for a, arch in enumerate(list(reversed(archetypes))):
    img = plt.imread(f"archetypes/{fp_type}/{arch['image_filename']}")
    value = fingerprint_data[a]
    offset_image(value, a, arch, bar_is_too_short=value < max_value / 10, ax=plt.gca())
    
plt.subplots_adjust(left=0.15)

plt.xlim(0, max(fingerprint_data) * 1.10)
plt.ylim(-0.5, len(descriptions) - 0.5)
plt.tight_layout()

plt.show()
